In [7]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re

from IPython.display import display, clear_output

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf

ModuleNotFoundError: No module named 'pkg_resources'

In [ ]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [ ]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = 4628
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [ ]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

In [ ]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                return result
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['ATR'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def average_close_levels(significant_values, percentage_threshold=0.001):
    if len(significant_values) == 0:
        return np.array([])

    # Sort the significant values
    sorted_values = np.sort(significant_values)

    # Initialize lists to store averaged values
    averaged_values = []
    i = 0
    while i < len(sorted_values):
        current_value = sorted_values[i]
        group = [current_value]

        # Find all values within the percentage threshold
        j = i + 1
        while j < len(sorted_values) and abs((sorted_values[j] - current_value) / current_value) <= percentage_threshold:
            group.append(sorted_values[j])
            j += 1

        # Calculate the average of the grouped values
        average_value = np.mean(group)
        average_value = round(average_value, 2)
        averaged_values.append(average_value)

        # Move to the next group
        i = j

    return np.array(averaged_values)

def find_nearest_significant(value, significant_values):
    significant_values = np.array(significant_values)
    nearest_value = significant_values[np.argmin(np.abs(significant_values - value))]
    return nearest_value

In [ ]:
ist = timezone('Asia/Kolkata')

def process_df_with_features(df):
    df['datetime'] = pd.to_datetime(df['datetime'], unit='s')

    df['datetime'] = df['datetime'].dt.tz_localize('UTC').dt.tz_convert(ist).dt.tz_localize(None)

    df.set_index(df['datetime'], inplace=True)

    df.drop('datetime', axis=1, inplace=True)
    df.drop('volume', axis=1, inplace=True)

    df['hour_of_day'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek

    df['high_low_range'] = df['high'] - df['low']
    df['open_close_range'] = df['open'] - df['close']

    df['EMA'] = ta.ema(df['close'])
    df['SMA'] = ta.sma(df['close'])
    df['WMA'] = ta.wma(df['close'])

    macd = ta.macd(df['close'])
    df = pd.concat([df, macd], axis=1)

    df['RSI'] = ta.rsi(df['close'])

    bollinger_bands = ta.bbands(df['close'])
    df = pd.concat([df, bollinger_bands], axis=1)

    stoch_oscillator = ta.stoch(df['high'], df['low'], df['close'])
    df = pd.concat([df, stoch_oscillator], axis=1)

    df['ATR'] = ta.atr(df['high'], df['low'], df['close'])
    df['CCI'] = ta.cci(df['high'], df['low'], df['close'])
    df['ROC'] = ta.roc(df['close'])
    df['MOM'] = ta.mom(df['close'])
    df['WPR'] = ta.willr(df['high'], df['low'], df['close'])
    df['Z_Score'] = ta.zscore(df['close'])

    significant_max, significant_min = find_local_extrema(df)
    df['Significant_Max'] = 0.0
    df['Significant_Min'] = 0.0

    for idx in significant_max:
        df.at[df.index[idx], 'Significant_Max'] = df.at[df.index[idx], 'high']

    for idx in significant_min:
        df.at[df.index[idx], 'Significant_Min'] = df.at[df.index[idx], 'low']

    df = df.round(2)

    df.dropna(inplace=True)

    return df

In [ ]:
train_candles = fetch_candle_data(100)

train_df = pd.DataFrame(train_candles['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

#train_df = fetch_train_candle_data(10)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

In [ ]:
train_df = process_df_with_features(train_df)
train_df

In [ ]:
significant_values = np.concatenate([
        train_df['Significant_Max'].loc[train_df['Significant_Max'] != 0].values,
        train_df['Significant_Min'].loc[train_df['Significant_Min'] != 0].values
    ])

significant_values

In [ ]:
averaged_significant_values = average_close_levels(significant_values)

len(significant_values), len(averaged_significant_values)

In [ ]:
#nearest_level = find_nearest_significant(51300, averaged_significant_values)

#nearest_level

In [ ]:
def format_capital(capital):
    # Check if the value is negative
    negative = capital < 0
    # Use the absolute value for formatting
    capital = abs(capital)

    if capital >= 1_00_00_00_00_00_00_000:  # 1 Shankh
        formatted_value = f'{capital / 1_00_00_00_00_00_00_000:.2f} Shankh'
    elif capital >= 1_00_00_00_00_00_00_000:  # 1 Padma
        formatted_value = f'{capital / 1_00_00_00_00_00_00_000:.2f} Padma'
    elif capital >= 1_00_00_00_00_00_000:  # 1 Nil
        formatted_value = f'{capital / 1_00_00_00_00_00_000:.2f} Nil'
    elif capital >= 1_00_00_00_00_000:  # 1 Kharab
        formatted_value = f'{capital / 1_00_00_00_00_000:.2f} Kharab'
    elif capital >= 1_00_00_00_000:  # 1 Arab
        formatted_value = f'{capital / 1_00_00_00_000:.2f} Arab'
    elif capital >= 1_00_00_000:  # 1 Crore
        formatted_value = f'{capital / 1_00_00_000:.2f} Cr'
    elif capital >= 1_00_000:  # 1 Lakh
        formatted_value = f'{capital / 1_00_000:.2f} L'
    elif capital >= 1_000:  # 1 Thousand
        formatted_value = f'{capital / 1_000:.2f} K'
    else:
        formatted_value = f'{capital:.2f}'

    # Prepend the negative sign if needed
    return f'-{formatted_value}' if negative else formatted_value

In [ ]:
# Function to format trade time
def format_trade_time(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)

    if hours > 0:
        return f"{int(hours)} Hour{'s' if hours > 1 else ''} {int(minutes)} Min{'s' if minutes > 1 else ''}"
    elif minutes > 0:
        return f"{int(minutes)} Min{'s' if minutes > 1 else ''}"
    else:
        return f"{int(seconds)} Sec{'S' if seconds > 1 else ''}"

In [ ]:
backtest_candle_data = fetch_candle_data(10)

raw_backtest_df = pd.DataFrame(backtest_candle_data['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

print(len(raw_backtest_df))

raw_backtest_df = raw_backtest_df.drop_duplicates(subset='datetime', keep='first')
raw_backtest_df

print(len(raw_backtest_df))

In [ ]:
initial_capital = 10000
backtest_results = []

crop_fetched_candle_data = 532

def backtest_logic():
    global backtest_results

    backtest_capital = initial_capital
    backtest_quantity = quantity
    temp_capital = initial_capital

    total_backtest_profits = 0
    total_backtest_losses = 0
    total_backtest_brokerage = 0

    backtest_trade_active = False
    backtest_entry_price = 0
    backtest_target_price = 0
    backtest_stop_loss_price = 0

    entry_type = None
    backtest_entry_datetime = None
    backtest_levels = None

    detected_patterns = []

    for i in range(crop_fetched_candle_data, len(raw_backtest_df)):
        if backtest_capital > 2 * temp_capital:
            backtest_quantity *= 2
            temp_capital *= 2

        backtest_df = raw_backtest_df[i-crop_fetched_candle_data:i+1].copy()

        backtest_df = process_df_with_features(backtest_df)

        print(len(backtest_df))

        backtest_df = backtest_df.iloc[:-1]

        # Actual Closing Price
        actual_price = backtest_df['close'].iloc[-1]
        actual_high = backtest_df['high'].iloc[-1]
        actual_low = backtest_df['low'].iloc[-1]

        current_time = backtest_df.index[-1].time()

        backtest_significant_values = np.concatenate([
                backtest_df['Significant_Max'].loc[backtest_df['Significant_Max'] != 0].values,
                backtest_df['Significant_Min'].loc[backtest_df['Significant_Min'] != 0].values
            ])

        backtest_average_values = average_close_levels(backtest_significant_values)
        backtest_nearest_level = find_nearest_significant(actual_price, backtest_significant_values)

        print(backtest_df.index[-1])
        print(f'Capital: {format_capital(backtest_capital)}')

        if current_time >= dt_time(9, (15 + interval_minutes)) and current_time <= dt_time(15, 0):
            if not backtest_trade_active and entry_type == None:
                curr_atr = backtest_df['ATR'].iloc[-1]
                curr_momentum = backtest_df['MOM'].iloc[-1]
                curr_rsi = backtest_df['RSI'].iloc[-1]

                if (actual_price >= backtest_nearest_level and curr_rsi <= 50 and
                    (curr_momentum < 0)):

                    backtest_entry_price = actual_price

                    backtest_target_price = int(actual_price + curr_atr)
                    backtest_stop_loss_price = int(actual_price - curr_atr)

                    backtest_trade_active = True
                    entry_type = "CE"
                    backtest_entry_datetime = backtest_df.index[-1]
                    backtest_levels = backtest_nearest_level

                    print(f"CE Signal at {backtest_df.index[-1]}")
                    print(f"Entry Price: {backtest_entry_price}")
                    print(f"Target: {backtest_target_price}")
                    print(f"Stop Loss: {backtest_stop_loss_price}")

                elif (actual_price <= backtest_nearest_level and curr_rsi >= 50 and
                    (curr_momentum > 0)):

                    backtest_entry_price = actual_price

                    backtest_target_price = int(actual_price - curr_atr)
                    backtest_stop_loss_price = int(actual_price + curr_atr)

                    backtest_trade_active = True
                    entry_type = "PE"
                    backtest_entry_datetime = backtest_df.index[-1]
                    backtest_levels = backtest_nearest_level

                    print(f"PE Signal at {backtest_df.index[-1]}")
                    print(f"Entry Price: {backtest_entry_price}")
                    print(f"Target: {backtest_target_price}")
                    print(f"Stop Loss: {backtest_stop_loss_price}")

            else:
                if entry_type == "CE":
                    if actual_high >= backtest_target_price:
                        points = int((backtest_target_price - backtest_entry_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_profits += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"CE Target hit at {backtest_df.index[-1]}")
                        print(f"Points: {points}")
                        print(f"Profit: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_price,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),
                            'Support / Resistance': backtest_levels,
                            'Detected Patterns': detected_patterns
                        })

                        backtest_trade_active = False
                        entry_type = None
                        detected_patterns = []

                    elif actual_low <= backtest_stop_loss_price:
                        points = int((backtest_stop_loss_price - backtest_entry_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_losses += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"CE SL hit at {backtest_df.index[-1]}")
                        print(f"Points: {points}")
                        print(f"Loss: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_price,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),
                            'Support / Resistance': backtest_levels,
                            'Detected Patterns': detected_patterns
                        })

                        backtest_trade_active = False
                        entry_type = None
                        detected_patterns = []

                elif entry_type == "PE":
                    if actual_low <= backtest_target_price:
                        points = int((backtest_entry_price - backtest_target_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_profits += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"PE Target hit at {backtest_df.index[-1]}")
                        print(f"Points: {points}")
                        print(f"Profit: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_price,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),
                            'Support / Resistance': backtest_levels,
                            'Detected Patterns': detected_patterns
                        })

                        backtest_trade_active = False
                        entry_type = None
                        detected_patterns = []

                    elif actual_high >= backtest_stop_loss_price:
                        points = int((backtest_entry_price - backtest_stop_loss_price))
                        profits = int(points * backtest_quantity)
                        backtest_capital += profits
                        total_backtest_losses += 1
                        total_backtest_brokerage += brokerage

                        win_percentage = round(total_backtest_profits / (total_backtest_profits + total_backtest_losses) * 100, 2)
                        loss_percentage = round(total_backtest_losses / (total_backtest_profits + total_backtest_losses) * 100, 2)

                        print(f"PE SL hit at {backtest_df.index[-1]}")
                        print(f"Points: {points}")
                        print(f"Loss: {format_capital(profits)}")
                        print(f'Profit/Loss %: {win_percentage}% / {loss_percentage}%')

                        backtest_results.append({
                            'Entry Time': backtest_entry_datetime,
                            'Exit Time': backtest_df.index[-1],
                            'Entry Type': entry_type,
                            'Entry Price': backtest_entry_price,
                            'Exit Price': actual_price,
                            'Quantity': backtest_quantity,
                            'Points': points,
                            'Profit': format_capital(profits),
                            'Accuracy': f"{win_percentage}%",
                            'Capital': format_capital(backtest_capital),
                            'Brokerage': format_capital(total_backtest_brokerage),
                            'Support / Resistance': backtest_levels,
                            'Detected Patterns': detected_patterns
                        })

                        backtest_trade_active = False
                        entry_type = None
                        detected_patterns = []

        else:
            backtest_trade_active = False
            entry_type = None
            detected_patterns = []

        clear_output(wait=True)

backtest_logic()

In [ ]:
backtest_results_df = pd.DataFrame(backtest_results)

backtest_results_df.tail(15)

In [ ]:
# Step 1: Create an indicator for positive and negative values
backtest_results_df['Sign'] = backtest_results_df['Points'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

# Step 2: Identify streaks
backtest_results_df['Streak'] = (backtest_results_df['Sign'] != backtest_results_df['Sign'].shift()).cumsum()
backtest_results_df['Streak_Length'] = backtest_results_df.groupby(['Sign', 'Streak'])['Points'].transform('size')

# Step 3: Find the maximum streaks
max_positive_streak = backtest_results_df[backtest_results_df['Sign'] == 'Positive']['Streak_Length'].max()
max_negative_streak = backtest_results_df[backtest_results_df['Sign'] == 'Negative']['Streak_Length'].max()

print("Highest positive streak:", max_positive_streak)
print("Highest negative streak:", max_negative_streak)

In [ ]:
def parse_capital(formatted_value):
    suffixes = {
        'Shankh': 1_00_00_00_00_00_00_000,
        'Padma': 1_00_00_00_00_00_00_000,
        'Nil': 1_00_00_00_00_00_000,
        'Kharab': 1_00_00_00_00_000,
        'Arab': 1_00_00_00_000,
        'Cr': 1_00_00_000,
        'L': 1_00_000,
        'K': 1_000
    }

    for suffix, multiplier in suffixes.items():
        if formatted_value.endswith(suffix):
            return float(formatted_value.replace(suffix, '').strip()) * multiplier

    return float(formatted_value)

# Convert the columns back to float
backtest_results_df['Inv_Capital'] = backtest_results_df['Capital'].apply(parse_capital)

In [ ]:
def currency(x, pos):
    return f'₹{format_capital(x)}'

# Calculate drawdown
backtest_results_df['Drawdown'] = backtest_results_df['Inv_Capital'].cummax() - backtest_results_df['Inv_Capital']
backtest_results_df['Drawdown Percent'] = (backtest_results_df['Drawdown'] / backtest_results_df['Inv_Capital'].cummax()) * 100

backtest_results_df['Temp_Profit'] = backtest_results_df['Inv_Capital'] - backtest_results_df['Drawdown']

# Plot drawdown
plt.style.use('dark_background')
plt.figure(figsize=(14, 14))

formatter = FuncFormatter(currency)

plt.subplot(2, 1, 1)
plt.plot(backtest_results_df.index, backtest_results_df['Inv_Capital'], color='green', label='Capital')
plt.ylabel('Capital', color='white')
plt.title('Capital Over Time', color='white')
plt.legend()
plt.grid(False)
plt.tick_params(colors='white')
plt.gca().yaxis.set_major_formatter(formatter)

plt.subplot(2, 1, 2)
plt.plot(backtest_results_df.index, -backtest_results_df['Drawdown'], label='Drawdown', color='red')
plt.ylabel('Drawdown', color='white')
plt.title('Drawdown Over Time', color='white')
plt.legend()
plt.grid(False)
plt.tick_params(colors='white')
plt.gca().yaxis.set_major_formatter(formatter)

plt.tight_layout()
plt.show()

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15):
    now = datetime.now()
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before the market starts, set next_run_time to market start time
        next_run_time = market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(rf_final_pred, actual_closing_price, predicted_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_significant_values = np.concatenate([
            final_df['Significant_Max'].loc[final_df['Significant_Max'] != 0].values,
            final_df['Significant_Min'].loc[final_df['Significant_Min'] != 0].values
        ])

    final_average_values = average_close_levels(final_significant_values)
    final_nearest_level = find_nearest_significant(actual_closing_price, final_significant_values)

    final_current_time = final_df.index[-1].time()
    final_current_momentum = final_df['MOM'].iloc[-1]
    final_current_rsi = final_df['RSI'].iloc[-1]

    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(15, 0):
        #CE entry
        if (actual_closing_price >= final_nearest_level and final_current_rsi <= 50 and
            (final_current_momentum <= 0 and (
                final_df['Engulfing'].iloc[-1] == 100 or
                final_df['Hammer'].iloc[-1] != 0 or
                final_df['Morning_Star'].iloc[-1] != 0 or
                final_df['Inverted_Hammer'].iloc[-1] != 0
            ))):
            if not active_order:
                pygame.mixer.music.play()

                sl_hit_condition = False
                unsubscribe_done = False

                print("Making CE Position")
                ce_entry()

        #PE entry
        elif (actual_closing_price <= final_nearest_level and final_current_rsi >= 50 and
            (final_current_momentum >= 0 and (
                final_df['Engulfing'].iloc[-1] == -100 or
                final_df['Hanging_Man'].iloc[-1] != 0 or
                final_df['Evening_Star'].iloc[-1] != 0 or
                final_df['Shooting_Star'].iloc[-1] != 0
            ))):
            if not active_order:
                pygame.mixer.music.play()

                sl_hit_condition = False
                unsubscribe_done = False

                print("Making PE Position")
                pe_entry()

In [ ]:
def fetch_and_prepare_final_data():
    final_candle_data = fetch_candle_data(10)

    final_df = pd.DataFrame(final_candle_data['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])

    final_df = final_df.drop_duplicates(subset='datetime', keep='first')

    final_df = final_df[-(crop_fetched_candle_data + 1):]

    final_df = process_df_with_features(final_df)

    return final_df

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(point1, len(df)))
    full_trendline = [np.nan] * point1 + list(trendline)
    return full_trendline

In [ ]:
#while True:
clear_output(wait=True)

final_df = fetch_and_prepare_final_data()
print(len(final_df))

num_candles = 99

final_df = final_df.iloc[:-1]

target = final_df['ATR'].iloc[-1]

trailing_sl = final_df['ATR'].iloc[-1]

# Identify most recent high and low points
recent_highs, recent_lows = find_local_extrema(final_df)

most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

high_trendline = [np.nan] * len(final_df)
low_trendline = [np.nan] * len(final_df)

if most_recent_high is not None:
    previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
    high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

if most_recent_low is not None:
    previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
    low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

# Prepare candlestick data for mplfinance
actual_candles = final_df[-num_candles:].copy()

# Create a DataFrame for mplfinance
mpf_df = actual_candles[['open', 'high', 'low', 'close']]

# Create addplot elements for predicted prices and actual close prices
ap = [
    mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
    #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
]

# Add trendlines to the plot
if most_recent_high is not None:
    ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

if most_recent_low is not None:
    ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

# Add significant levels to the plot with conditional coloring
last_closing_price = final_df['close'].iloc[-1]

final_significant_values = np.concatenate([
        final_df['Significant_Max'][-num_candles:].loc[final_df['Significant_Max'] != 0].values,
        final_df['Significant_Min'][-num_candles:].loc[final_df['Significant_Min'] != 0].values
    ])

final_average_values = average_close_levels(final_significant_values)

for level in final_average_values:
    if level != 0:
        color = 'red' if level > last_closing_price else 'green'
        ap.append(mpf.make_addplot([level] * num_candles, color=color, linestyle='-', panel=0, secondary_y=False))

fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

for ax in axlist:
    ax.grid(False)

axlist[0].legend()

plt.show()

#market_entry_exit_logic(rf_final_pred, final_df['close'].iloc[-1], y_pred_ensemble_latest_original, final_df)

#sleep_time = get_sleep_time(interval_minutes)
#time.sleep(sleep_time)